In [ ]:
# ==========================================================
# LOAD VERSION 1 PHYSICS FEATURES
# ==========================================================

physics_v1 = pd.read_csv(

    "/content/drive/MyDrive/Exoplanet_Dataset/10_Physics_Features/Physics_Features.csv"

)

print("="*60)
print("VERSION 1 PHYSICS")
print("="*60)

print()

print("Rows :", len(physics_v1))

print("Columns :", len(physics_v1.columns))

display(physics_v1.head())

VERSION 1 PHYSICS

Rows : 1172
Columns : 8


,TIC_ID,ingress_fraction,flat_bottom_fraction,transit_snr,odd_even_ratio,secondary_depth,secondary_snr,duty_cycle
0,737546,0.316427,0.367147,4.386221,1.096883,30812.446592,0.938785,0.093915
1,927579,0.140704,0.718592,0.868916,NaN,403.933092,0.056066,0.020348
2,1528696,0.385253,0.229493,0.381656,1.010677,417.090013,0.025352,0.037364
3,1950736,0.431389,0.137221,0.107664,5.257057,0.000000,0.000000,0.312868
4,2234723,0.439327,0.121346,2.430875,1.240329,0.000000,0.000000,0.446803


In [ ]:
# ==========================================================
# BUILD VERSION 2 DATASET
# ==========================================================

physics_v2 = physics_v1.copy()

print()

print("Starting Rows :", len(physics_v2))


Starting Rows : 1172


In [ ]:
# ==========================================================
# NEW VERSION 2 FEATURES
# ==========================================================

NEW_COLUMNS = [

    "oot_rms",

    "transit_rms",

    "shape_rmse",

    "transit_skewness",

    "transit_kurtosis",

    "transit_mad",

    "transit_iqr"

]

for column in NEW_COLUMNS:

    physics_v2[column] = np.nan

physics_v2.head()

,TIC_ID,ingress_fraction,flat_bottom_fraction,transit_snr,odd_even_ratio,secondary_depth,secondary_snr,duty_cycle,oot_rms,transit_rms,shape_rmse,transit_skewness,transit_kurtosis,transit_mad,transit_iqr
0,737546,0.316427,0.367147,4.386221,1.096883,30812.446592,0.938785,0.093915,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,927579,0.140704,0.718592,0.868916,NaN,403.933092,0.056066,0.020348,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1528696,0.385253,0.229493,0.381656,1.010677,417.090013,0.025352,0.037364,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1950736,0.431389,0.137221,0.107664,5.257057,0.000000,0.000000,0.312868,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2234723,0.439327,0.121346,2.430875,1.240329,0.000000,0.000000,0.446803,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# ==========================================================
# COMPUTE NEW FEATURES
# ==========================================================

def process_star_v2(row):

    lightcurve = load_lightcurve(row)

    phase, flux, flux_error, transit_mask, duration_phase = prepare_folded_lightcurve(

        row,

        lightcurve

    )

    local = extract_local_variability_features(

        flux,

        transit_mask

    )

    symmetry = extract_depth_symmetry(

        phase,

        flux,

        transit_mask

    )

    stats = extract_transit_statistics(

        flux,

        transit_mask

    )

    return {

        "oot_rms": local["oot_rms"],

        "transit_rms": local["transit_rms"],

        "shape_rmse": symmetry["shape_rmse"],

        "transit_skewness": stats["transit_skewness"],

        "transit_kurtosis": stats["transit_kurtosis"],

        "transit_mad": stats["transit_mad"],

        "transit_iqr": stats["transit_iqr"]

    }

In [ ]:
# ==========================================================
# GENERATE VERSION 2 FEATURES
# ==========================================================

print("=" * 60)
print("GENERATING VERSION 2 FEATURES")
print("=" * 60)

master_lookup = dataset.copy()

failed = []

for index in tqdm(physics_v2.index):

    tic = physics_v2.loc[index, "TIC_ID"]

    try:

        row = master_lookup[
        master_lookup["TIC_ID"] == tic
        ].iloc[0]

        features = process_star_v2(row)

        for key, value in features.items():

            physics_v2.loc[index, key] = value

    except Exception as e:

        failed.append(tic)

        print(f"Failed TIC {tic}")
        print(e)

GENERATING VERSION 2 FEATURES


100%|██████████| 1172/1172 [19:48<00:00,  1.01s/it]


In [ ]:
tic = physics_v2.iloc[0]["TIC_ID"]

row = master_lookup.loc[tic]

print(type(row))
print(row.index.tolist())

<class 'pandas.core.series.Series'>
['Label', 'Number_of_Sectors', 'Best_Sector', 'period_days', 'duration_days', 'epoch_days', 'bls_depth', 'bls_power', 'peak_sigma']


In [ ]:
# ==========================================================
# SAVE PHYSICS FEATURES V2
# ==========================================================

OUTPUT_FILE = "/content/drive/MyDrive/Exoplanet_Dataset/10_Physics_Features_V2/Physics_Features_V2.csv"

physics_v2.to_csv(

    OUTPUT_FILE,

    index=False

)

print("=" * 60)
print("PHYSICS FEATURES V2 SAVED")
print("=" * 60)

print()

print("Saved to:")
print(OUTPUT_FILE)

print()

print("Rows :", len(physics_v2))
print("Columns :", len(physics_v2.columns))

PHYSICS FEATURES V2 SAVED

Saved to:
/content/drive/MyDrive/Exoplanet_Dataset/10_Physics_Features_V2/Physics_Features_V2.csv

Rows : 1172
Columns : 15


In [ ]:
# ==========================================================
# PHYSICS FEATURES V2 SUMMARY
# ==========================================================

print("=" * 60)
print("PHYSICS FEATURES V2")
print("=" * 60)

print()

print("Rows :", len(physics_v2))
print("Columns :", len(physics_v2.columns))

print()

display(physics_v2.head())

print()

print("Missing Values")
print()

print(physics_v2.isna().sum())

print()

print("Duplicate TIC IDs :", physics_v2["TIC_ID"].duplicated().sum())

print()

display(

    physics_v2.describe().T

)

PHYSICS FEATURES V2

Rows : 1172
Columns : 15



,TIC_ID,ingress_fraction,flat_bottom_fraction,transit_snr,odd_even_ratio,secondary_depth,secondary_snr,duty_cycle,oot_rms,transit_rms,shape_rmse,transit_skewness,transit_kurtosis,transit_mad,transit_iqr
0,737546,0.316427,0.367147,4.386221,1.096883,30812.446592,0.938785,0.093915,0.036868,0.049457,0.039887,0.983789,0.216492,0.028651,0.074134
1,927579,0.140704,0.718592,0.868916,NaN,403.933092,0.056066,0.020348,0.008708,0.010985,0.012718,0.126679,-0.190785,0.007628,0.015422
2,1528696,0.385253,0.229493,0.381656,1.010677,417.090013,0.025352,0.037364,0.016637,0.016851,0.018638,0.022705,0.368479,0.010459,0.020941
3,1950736,0.431389,0.137221,0.107664,5.257057,0.000000,0.000000,0.312868,0.001061,0.001057,0.001443,0.050365,0.042025,0.000703,0.001403
4,2234723,0.439327,0.121346,2.430875,1.240329,0.000000,0.000000,0.446803,0.001804,0.002026,0.002974,-0.037271,0.225180,0.001310,0.002624



Missing Values

TIC_ID                    0
ingress_fraction          0
flat_bottom_fraction      0
transit_snr              12
odd_even_ratio          137
secondary_depth          21
secondary_snr            21
duty_cycle                0
oot_rms                   0
transit_rms               0
shape_rmse                1
transit_skewness          0
transit_kurtosis          0
transit_mad               0
transit_iqr               0
dtype: int64

Duplicate TIC IDs : 0



,count,mean,std,min,25%,50%,75%,max
TIC_ID,1172.0,2.332153e+08,1.591498e+08,737546.000000,1.229387e+08,2.302209e+08,3.408830e+08,1.965257e+09
ingress_fraction,1172.0,2.647820e-01,1.220605e-01,0.025000,1.610681e-01,2.895931e-01,3.651340e-01,9.412496e-01
flat_bottom_fraction,1172.0,4.732182e-01,2.354031e-01,0.000000,2.697320e-01,4.208138e-01,6.778639e-01,9.500000e-01
transit_snr,1160.0,2.918503e+00,3.528645e+00,0.019568,5.705948e-01,1.495258e+00,4.233704e+00,3.542350e+01
odd_even_ratio,1035.0,8.045329e+01,2.487786e+03,-0.209211,9.448863e-01,1.006150e+00,1.097637e+00,8.003349e+04
secondary_depth,1151.0,9.723971e+03,3.953938e+04,0.000000,0.000000e+00,1.294468e+02,7.795609e+02,7.388970e+05
secondary_snr,1151.0,3.085304e-01,6.072511e-01,0.000000,0.000000e+00,4.829440e-02,2.171607e-01,4.741744e+00
duty_cycle,1172.0,8.648123e-02,1.122681e-01,0.002503,2.197991e-02,3.905722e-02,9.683034e-02,6.792986e-01
oot_rms,1172.0,2.381764e-02,2.604283e-01,0.000255,1.683699e-03,3.802117e-03,1.267661e-02,8.767529e+00
transit_rms,1172.0,3.990621e-02,5.149516e-01,0.000243,2.158086e-03,5.210129e-03,1.819896e-02,1.749352e+01
